In [1]:
library(ggplot2)
library(reshape2)
library(RColorBrewer)
library(dplyr)
library(gridExtra)
library(cowplot)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine




In [3]:
metafile <- '../../../../hybrid_diversity/5_snp_analyses/6_hybrid_analyses/metadata.txt'
admixture_dir <- '../../../../hybrid_diversity/5_snp_analyses/6_hybrid_analyses/admixture_runs/'

In [4]:
# Read metadata
metadata <- read.csv(metafile, sep = "\t", stringsAsFactors = FALSE, header = TRUE)

In [5]:
# Read .fam file to get ADMIXTURE sample order
fam_file <- "../../../../hybrid_diversity/5_snp_analyses/6_hybrid_analyses/hybrid_strains_pruned.fam"
if (!file.exists(fam_file)) {
  stop("hybrid_strains_pruned.fam not found. Check file path.")
}
fam_data <- read.table(fam_file, header = FALSE, stringsAsFactors = FALSE)
admixture_order <- fam_data$V2  # Sample IDs from PLINK/ADMIXTURE
cat("ADMIXTURE analysis included", length(admixture_order), "samples\n")
 
# Check for sample mismatches
missing_in_metadata <- setdiff(admixture_order, metadata$strain_ID)
missing_in_admixture <- setdiff(metadata$strain_ID, admixture_order)
 
if (length(missing_in_metadata) > 0) {
  cat("WARNING: Samples in ADMIXTURE but not in metadata:", paste(missing_in_metadata, collapse = ", "), "\n")
}
if (length(missing_in_admixture) > 0) {
  cat("NOTE: Samples in metadata but not in ADMIXTURE:", paste(missing_in_admixture, collapse = ", "), "\n")
}


ADMIXTURE analysis included 22 samples


In [6]:
# Function to read ADMIXTURE Q file
read_admixture_q_corrected <- function(qfile, k) {
  if (!file.exists(qfile)) {
    cat("WARNING: File not found:", qfile, "\n")
    return(NULL)
  }
  
  # Read Q matrix
  q_data <- read.table(qfile, header = FALSE, stringsAsFactors = FALSE)
  
  # Check dimensions
  if (nrow(q_data) != length(admixture_order)) {
    stop("Mismatch: Q file has ", nrow(q_data), " rows but .fam has ", length(admixture_order), " samples")
  }
  if (ncol(q_data) != k) {
    stop("Mismatch: Q file has ", ncol(q_data), " columns but expected K=", k)
  }
  
  # Add proper column names and sample IDs
  colnames(q_data) <- paste0("Pop", 1:k)
  q_data$Sample <- admixture_order
  q_data$K <- k
  
  # Merge with metadata using explicit matching
  q_merged <- merge(q_data, metadata, by.x = "Sample", by.y = "strain_ID", all.x = TRUE)
  
  # Check for failed merges
  failed_merges <- sum(is.na(q_merged$cytotype))
  if (failed_merges > 0) {
    cat("WARNING:", failed_merges, "samples failed to merge with metadata for K=", k, "\n")
    cat("Failed samples:", q_merged$Sample[is.na(q_merged$cytotype)], "\n")
  }
  
  return(q_merged)
}


In [14]:
# STEP 1: Find the best replicate for each K based on CV error
cat("\n===== STEP 1: Identifying best replicate per K =====\n")

best_reps <- list()  # Will store: best_reps[[k]] = list(rep = best_rep_num, cv_error = cv_value)
all_cv_errors <- list()  # Will store all CV errors for plotting

for (k in 1:12) {
  cv_errors_k <- c()
  rep_found <- FALSE
  
  # Try to find log files for this K
  for (rep in 1:10) {
    logfile <- paste0(admixture_dir, "admixture_K", k, "_rep", rep, ".log")
    
    if (file.exists(logfile)) {
      # Extract CV error from log
      cv_line <- system(paste("grep 'CV error' ", logfile), intern = TRUE)
      if (length(cv_line) > 0) {
        cv_error <- as.numeric(gsub(".*CV error.*: ([0-9.]+).*", "\\1", cv_line))
        cv_errors_k <- c(cv_errors_k, cv_error)
        rep_found <- TRUE
      }
    }
  }
  
  # If we found CV errors for this K, identify the best one
  if (length(cv_errors_k) > 0) {
    best_rep_idx <- which.min(cv_errors_k)
    best_rep_num <- best_rep_idx  # rep numbering starts at 1
    best_cv <- cv_errors_k[best_rep_idx]
    
    best_reps[[as.character(k)]] <- list(rep = best_rep_num, cv_error = best_cv)  # KEY: use as.character(k) as name
    all_cv_errors[[as.character(k)]] <- cv_errors_k
    
    cat("K=", k, ": Best rep =", best_rep_num, " (CV error =", round(best_cv, 4), ", found", length(cv_errors_k), "replicates)\n")
  }
}

cat("\nCompleted K values:", paste(names(best_reps), collapse = ", "), "\n")


===== STEP 1: Identifying best replicate per K =====
K= 1 : Best rep = 1  (CV error = 0.255 , found 10 replicates)
K= 2 : Best rep = 4  (CV error = 0.2557 , found 10 replicates)
K= 3 : Best rep = 4  (CV error = 0.273 , found 10 replicates)
K= 4 : Best rep = 8  (CV error = 0.2712 , found 10 replicates)
K= 5 : Best rep = 3  (CV error = 0.345 , found 10 replicates)
K= 6 : Best rep = 3  (CV error = 0.2977 , found 10 replicates)
K= 7 : Best rep = 5  (CV error = 0.2915 , found 10 replicates)
K= 8 : Best rep = 10  (CV error = 0.3363 , found 10 replicates)
K= 9 : Best rep = 2  (CV error = 0.3981 , found 10 replicates)
K= 10 : Best rep = 4  (CV error = 0.5047 , found 10 replicates)
K= 11 : Best rep = 6  (CV error = 0.4812 , found 10 replicates)
K= 12 : Best rep = 7  (CV error = 0.212 , found 10 replicates)

Completed K values: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12 


In [15]:
# STEP 2: Load the best Q matrix for each K
cat("\n===== STEP 2: Loading best Q matrices =====\n")

q_list <- list()

for (k in which(!sapply(best_reps, is.null))) {
  best_rep <- best_reps[[k]]$rep
  qfile <- paste0(admixture_dir, "hybrid_strains_pruned.K", k, "_rep", best_rep, ".Q")
  
  if (file.exists(qfile)) {
    q_list[[k]] <- read_admixture_q_corrected(qfile, k)
    cat("Loaded K=", k, " rep=", best_rep, "\n")
  } else {
    cat("WARNING: Q file not found for K=", k, " rep=", best_rep, ": ", qfile, "\n")
  }
}



===== STEP 2: Loading best Q matrices =====
Loaded K= 1  rep= 1 
Loaded K= 2  rep= 4 
Loaded K= 3  rep= 4 
Loaded K= 4  rep= 8 
Loaded K= 5  rep= 3 
Loaded K= 6  rep= 3 
Loaded K= 7  rep= 5 
Loaded K= 8  rep= 10 
Loaded K= 9  rep= 2 
Loaded K= 10  rep= 4 
Loaded K= 11  rep= 6 
Loaded K= 12  rep= 7 


In [16]:
# STEP 3: Find optimal K
cat("\n===== STEP 3: Finding optimal K =====\n")

# Extract K values and their corresponding CV errors
cv_errors_best_reps <- sapply(best_reps, function(x) x$cv_error)
k_values <- as.numeric(names(cv_errors_best_reps))

if (length(cv_errors_best_reps) > 0) {
  # Find the index of minimum CV error
  min_idx <- which.min(cv_errors_best_reps)
  optimal_k <- k_values[min_idx]
  optimal_cv <- cv_errors_best_reps[min_idx]
  
  cat("\nOptimal K based on best rep CV error:", optimal_k, 
      "(CV error:", round(optimal_cv, 4), ")\n")
  cat("K values tested:", paste(sort(k_values), collapse = ", "), "\n")
} else {
  optimal_k <- 2
  cat("\nNo CV errors found, using K=2 as default\n")
}


===== STEP 3: Finding optimal K =====

Optimal K based on best rep CV error: 12 (CV error: 0.212 )
K values tested: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12 


In [17]:
custom_order <- c("ARC66", "ARC81", "ARC82", "ARC83", "ARC84",  # Carolina strains
                  "UTEX2797", "UTEX2797-SC1", "UTEX2797-SC2", "UTEX2797-SC3", "UTEX2797-SC4",  # UTEX2797 group
                  "UTEX2827", "PK2298C6", "PK2300D3", "8020", # Mixed ancestry strains
                  "13A5", "8027", "PK2297D4", "PK2299B3",  # Other Texas
                  "12A1", "12A1-SC1", "12A1-SC2", "12A1-SC3" )  # 12A1 group

greys_12 <- colorRampPalette(RColorBrewer::brewer.pal(9, "Greys"))(12)
greys_12_categorical <- greys_12[c(12, 1, 7, 4, 10, 2, 9, 5, 11, 3, 8, 6)]

In [22]:
# STEP 4: Create CV boxplot and admixture plots (skip K=1)
cat("\n===== STEP 4: Creating plots =====\n")

plot_list <- list()

# CV error boxplot across all replicates
if (length(all_cv_errors) > 0) {
  # Prepare data for boxplot
  cv_all_df <- data.frame()
  
  for (k_name in names(all_cv_errors)) {
    k_num <- as.numeric(k_name)
    cv_errors_k <- all_cv_errors[[k_name]]
    cv_all_df <- rbind(cv_all_df, data.frame(
      K = k_num,
      CV_Error = cv_errors_k,
      Rep = 1:length(cv_errors_k)
    ))
  }
  
  p_cv <- ggplot(cv_all_df, aes(x = factor(K), y = CV_Error)) +
    geom_boxplot(outlier.shape = NA, fill = "lightgrey", alpha = 0.7) +
    geom_jitter(width = 0.2, alpha = 0.5, size = 2, color = "#333333") +
    geom_vline(xintercept = which(as.numeric(levels(factor(cv_all_df$K))) == optimal_k), 
               color = "red", linetype = "dashed", linewidth = 1) +
    labs(title = "ADMIXTURE CV Error Distribution Across 10 Replicates",
         x = "K (Populations)", 
         y = "Cross-Validation Error") +
    theme_minimal() +
    theme(plot.margin = margin(12, 12, 12, 12),
          plot.title = element_text(size = 10)) +
    annotate("text", x = which(as.numeric(levels(factor(cv_all_df$K))) == optimal_k) + 0.3, 
             y = max(cv_all_df$CV_Error) * 0.95, 
             label = paste("Optimal K =", optimal_k), color = "red", size = 3.5)
  
  plot_list[["cv"]] <- p_cv
  cat("Created CV boxplot\n")
}

# Admixture plots for each K (skip K=1)
k_range <- which(!sapply(q_list, is.null))
k_range <- k_range[k_range > 1]  # Remove K=1

for (k in k_range) {
  if (!is.null(q_list[[k]])) {
    pop_cols <- paste0("Pop", 1:k)
    q_plot <- q_list[[k]][, c("Sample", "cytotype", "region", pop_cols)]
    q_plot$Sample <- factor(q_plot$Sample, levels = custom_order)
    q_plot <- q_plot[order(q_plot$Sample), ]
    
    q_melt <- melt(q_plot, id.vars = c("Sample", "cytotype", "region"), 
                   measure.vars = pop_cols, variable.name = "Population", value.name = "Ancestry")
    q_meta <- unique(q_melt[, c("Sample", "cytotype", "region")])
    
    p <- ggplot(q_melt, aes(x = Sample, y = Ancestry, fill = Population)) +
      geom_bar(stat = "identity", width = 1) +
      geom_point(data = q_meta, 
                 aes(x = Sample, y = -0.05, shape = region, color = cytotype),
                 size = 3, inherit.aes = FALSE) +
      scale_fill_manual(values = greys_12_categorical) +
      scale_shape_manual(values = c("Carolinas" = 15, "Texas" = 17), name = "Geographic Region") +
      scale_color_manual(values = c("A1" = "#FF69B4", "A2" = "#1E90FF"), name = "Chloroplast Cytotype") +
      labs(title = paste("K =", k), x = NULL, y = "Ancestry") +
      ylim(-0.05, 1.05) +
      theme_minimal() +
      theme(axis.text.x = element_text(angle = 270, hjust = 0, vjust = 0.5, size = 6),
            axis.title.y = element_text(size = 7),
            plot.title = element_text(size = 8),
            panel.grid = element_blank(),
            legend.position = "none",
            plot.margin = margin(12, 12, 12, 12))
    
    plot_list[[paste0("k", k)]] <- p
    cat("Created plot for K=", k, "\n")
  }
}



===== STEP 4: Creating plots =====
Created CV boxplot
Created plot for K= 2 
Created plot for K= 3 
Created plot for K= 4 
Created plot for K= 5 
Created plot for K= 6 
Created plot for K= 7 
Created plot for K= 8 
Created plot for K= 9 
Created plot for K= 10 
Created plot for K= 11 
Created plot for K= 12 


In [23]:
# STEP 5: Generate PDF with shared legend
cat("\n===== STEP 5: Generating PDF =====\n")

p_legend_source <- plot_list[[2]] +  # Use first K plot for legend
  theme(legend.position = "bottom",
        legend.text = element_text(size = 7),
        legend.title = element_text(size = 8)) +
  guides(fill = "none",
         shape = guide_legend(title = "Geographic Region", order = 1),
         color = guide_legend(title = "Chloroplast Cytotype", order = 2))

shared_legend <- get_legend(p_legend_source)

# Build all plots into one flat list: CV first, then admixture panels
all_plots <- c(list(cv = plot_list[["cv"]]), 
               plot_list[grep("^k", names(plot_list))])

# Pad to even number for clean 2-col layout
if (length(all_plots) %% 2 != 0) {
  all_plots[["empty"]] <- ggplot() + theme_void()
}

# Calculate page height from number of rows
n_rows <- ceiling(length(all_plots) / 2)
page_height <- (n_rows * 2.5) + 0.8  # panels + legend

pdf("admixture_plots.pdf", width = 14, height = page_height)
grid.arrange(
  arrangeGrob(grobs = all_plots, ncol = 2),
  shared_legend,
  ncol = 1,
  heights = c(n_rows * 2.5, 0.8)
)

dev.off()
cat("PDF saved to admixture_plots.pdf\n")



===== STEP 5: Generating PDF =====


agg_record_17f12b7e854df 
                       2

PDF saved to admixture_plots.pdf


In [24]:
# STEP 6: Summary statistics
cat("\n===== SUMMARY =====\n")
cat("Total K values analyzed:", length(best_reps), "\n")
cat("Optimal K:", optimal_k, "\n")
cat("Best rep CV error for optimal K:", round(best_reps[[optimal_k]]$cv_error, 4), "\n")

# Show which replicate was best for each K
cat("\nBest replicate per K:\n")
for (k in sort(as.numeric(names(best_reps)))) {
  cat(sprintf("  K=%2d: rep %2d (CV = %.4f)\n", k, best_reps[[k]]$rep, best_reps[[k]]$cv_error))
}



===== SUMMARY =====
Total K values analyzed: 12 
Optimal K: 12 
Best rep CV error for optimal K: 0.212 

Best replicate per K:
  K= 1: rep  1 (CV = 0.2550)
  K= 2: rep  4 (CV = 0.2557)
  K= 3: rep  4 (CV = 0.2730)
  K= 4: rep  8 (CV = 0.2712)
  K= 5: rep  3 (CV = 0.3450)
  K= 6: rep  3 (CV = 0.2977)
  K= 7: rep  5 (CV = 0.2915)
  K= 8: rep 10 (CV = 0.3363)
  K= 9: rep  2 (CV = 0.3981)
  K=10: rep  4 (CV = 0.5047)
  K=11: rep  6 (CV = 0.4812)
  K=12: rep  7 (CV = 0.2120)
